In [27]:
import os
import base64
from pathlib import Path
from typing import List, Optional, Union

from dotenv import load_dotenv
from mistralai import Mistral

load_dotenv()

# --- Config ---
INPUT_DIR = Path("../data")       # where your PDFs are
OUTPUT_DIR = Path("../data/ocr_md")       # where Markdown files will be saved
MODEL_NAME = "mistral-ocr-latest"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [28]:
def b64_encode_pdf(pdf_path: Path) -> Optional[str]:
    """Return base64 string for a PDF, or None on error."""
    try:
        with open(pdf_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        print(f"Not found: {pdf_path}")
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
    return None

In [29]:
def extract_pages(obj: Union[dict, object]) -> List[dict]:
    """
    Robustly extract 'pages' from the OCR response whether it's a dict or an object.
    Each page should have a 'markdown' attribute or key.
    """
    # Try attribute access
    pages = getattr(obj, "pages", None)
    if pages is not None:
        return pages or []

    # Try dict access
    if isinstance(obj, dict):
        return obj.get("pages", []) or []

    return []

In [30]:
def get_markdown_from_pages(pages: List[object]) -> str:
    """Join all per-page markdown strings into a single markdown blob."""
    md_parts: List[str] = []
    for p in pages:
        # attribute or dict access for 'markdown'
        md = getattr(p, "markdown", None)
        if md is None and isinstance(p, dict):
            md = p.get("markdown")
        if md:
            md_parts.append(md)
    return "\n\n".join(md_parts)


In [31]:
def ocr_pdf_to_markdown(client: Mistral, pdf_path: Path) -> Optional[str]:
    """Run OCR on a single PDF and return full Markdown text (or None on failure)."""
    encoded = b64_encode_pdf(pdf_path)
    if not encoded:
        return None

    try:

        ocr_response = client.ocr.process(
            model=MODEL_NAME,
            document={
                "type": "document_url",
                "document_url": f"data:application/pdf;base64,{encoded}"
            },
            include_image_base64=False,  # set True if you actually need inline images as base64
        )
    except Exception as e:
        print(f"  OCR request failed for {pdf_path.name}: {e}")
        return None

    pages = extract_pages(ocr_response)
    full_md = get_markdown_from_pages(pages)

    if not full_md.strip():
        print(f"  No markdown extracted for {pdf_path.name}.")
        return None

    return full_md

In [32]:
def main():
    api_key = os.environ.get("MISTRAL_API_KEY")
    if not api_key:
        raise RuntimeError("Missing MISTRAL_API_KEY in environment.")

    client = Mistral(api_key=api_key)

    pdf_paths = sorted(INPUT_DIR.glob("*.pdf"))
    if not pdf_paths:
        print(f" No PDFs found in {INPUT_DIR.resolve()}")
        return

    print(f" Found {len(pdf_paths)} PDF(s) in {INPUT_DIR.resolve()}")

    for pdf_path in pdf_paths:
        md = ocr_pdf_to_markdown(client, pdf_path)
        if md is None:
            print("skipped (no output).")
            continue
        out_path = OUTPUT_DIR / f"{pdf_path.stem}.md"
        try:
            out_path.write_text(md, encoding="utf-8")

        except Exception as e:
            print(f"\n  Failed to write {out_path}: {e}")


if __name__ == "__main__":
    main()

 Found 2 PDF(s) in /Users/ryan/Desktop/Work/SIGHT/simple_eval/data
